# Compare DDPG and SAC

Load both saved checkpoints, evaluate them on the same fixed obstacle layout, record videos, and summarize comparable metrics.

## Imports and Paths

In [1]:
from pathlib import Path
import sys

import numpy as np
import torch
from gymnasium.wrappers import RecordEpisodeStatistics, RecordVideo
from IPython.display import Markdown, Video, display

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()
    if BASE_DIR.name != "Continuous_Diff_Drive":
        BASE_DIR = BASE_DIR / "Continuous_Diff_Drive"

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from diff_drive_agent import DiffDriveDDPGAgent, DiffDriveSACAgent
from diff_drive_env import DiffDriveEnv

BASE_DIR

c:\Users\39324\anaconda3\envs\naml_libraries\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


WindowsPath('c:/Users/39324/Desktop/NAML_RL_gym/Continuous_Diff_Drive')

## Shared Evaluation Configuration

In [2]:
DDPG_CHECKPOINT_PATH = BASE_DIR / "models" / "ddpg_checkpoint.pt"
SAC_CHECKPOINT_PATH = BASE_DIR / "models" / "sac_checkpoint.pt"
COMPARISON_VIDEO_DIR = BASE_DIR / "videos" / "comparison"
DDPG_NAME_PREFIX = "compare_ddpg_diff_drive_eval_fixed_obstacles_greedy"
SAC_NAME_PREFIX = "compare_sac_diff_drive_eval_fixed_obstacles_greedy"

OBSTACLES = [
    (2.0, 4.0, 3.0, 0.3),
    (5.0, 2.0, 0.3, 3.0),
    (7.0, 6.0, 1.5, 0.3),
]

ENV_KWARGS = dict(
    room_size       = (10.0, 10.0),
    obstacles       = OBSTACLES,
    random_obst     = False,
    robot_start     = (1.0, 1.0),
    goal_pos        = (8.5, 8.5),
    max_step        = 1000,
    n_lidar_rays    = 16,
    lidar_max_range = 5.0,
    robot_radius    = 0.3,
    dt              = 0.1,
    render_mode     = "rgb_array",
)

DDPG_KWARGS = dict(
    actor_lr     = 1e-4,
    critic_lr    = 1e-3,
    discount     = 0.95,
    tau          = 0.005,
    noise_std    = 0.2,
    noise_clip   = 0.5,
    batch_size   = 256,
    buffer_size  = 100_000,
    hidden_dim   = 128,
    warmup_steps = 1_000,
    device       = "cpu",
)

SAC_KWARGS = dict(
    actor_lr     = 3e-4,
    critic_lr    = 3e-4,
    alpha_lr     = 3e-4,
    discount     = 0.99,
    tau          = 0.005,
    batch_size   = 256,
    buffer_size  = 100_000,
    hidden_dim   = 64,
    warmup_steps = 5_000,
    device       = "cpu",
)

N_EPISODES = 3
COMPARISON_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_VIDEO_DIR

WindowsPath('c:/Users/39324/Desktop/NAML_RL_gym/Continuous_Diff_Drive/videos/comparison')

## Create Agents and Load Checkpoints

In [3]:
if not DDPG_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"No DDPG checkpoint found at {DDPG_CHECKPOINT_PATH}")
if not SAC_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"No SAC checkpoint found at {SAC_CHECKPOINT_PATH}")

ddpg_env = DiffDriveEnv(**ENV_KWARGS)
sac_env = DiffDriveEnv(**ENV_KWARGS)

ddpg_agent = DiffDriveDDPGAgent(env=ddpg_env, **DDPG_KWARGS)
sac_agent = DiffDriveSACAgent(env=sac_env, **SAC_KWARGS)

ddpg_checkpoint = torch.load(DDPG_CHECKPOINT_PATH, map_location=DDPG_KWARGS["device"])
ddpg_agent.actor.load_state_dict(ddpg_checkpoint["actor"])
ddpg_agent.critic.load_state_dict(ddpg_checkpoint["critic"])
ddpg_agent.actor_target.load_state_dict(ddpg_checkpoint["actor_target"])
ddpg_agent.critic_target.load_state_dict(ddpg_checkpoint["critic_target"])

sac_agent.load_checkpoint(SAC_CHECKPOINT_PATH, load_optimizers=False)

print(f"Loaded DDPG checkpoint from {DDPG_CHECKPOINT_PATH}")
print(f"Loaded SAC checkpoint from {SAC_CHECKPOINT_PATH}")

SAC checkpoint loaded from c:\Users\39324\Desktop\NAML_RL_gym\Continuous_Diff_Drive\models\sac_checkpoint.pt
Loaded DDPG checkpoint from c:\Users\39324\Desktop\NAML_RL_gym\Continuous_Diff_Drive\models\ddpg_checkpoint.pt
Loaded SAC checkpoint from c:\Users\39324\Desktop\NAML_RL_gym\Continuous_Diff_Drive\models\sac_checkpoint.pt


## Benchmark Helpers

In [4]:
def select_greedy_action(agent, obs, algorithm):
    if algorithm == "DDPG":
        return agent.select_action(obs, add_noise=False)
    if algorithm == "SAC":
        return agent.select_action(obs, evaluate=True)
    raise ValueError(f"Unknown algorithm: {algorithm}")

def run_recorded_benchmark(agent, algorithm, env_kwargs, video_folder, name_prefix, n_episodes):
    eval_env = DiffDriveEnv(**env_kwargs)
    eval_env = RecordVideo(
        eval_env,
        video_folder=str(video_folder),
        name_prefix=name_prefix,
        episode_trigger=lambda ep: True,
    )
    eval_env = RecordEpisodeStatistics(eval_env)

    rows = []
    for episode in range(n_episodes):
        obs, _ = eval_env.reset()
        done = False
        total_reward = 0.0
        info = {}

        while not done:
            action = select_greedy_action(agent, obs, algorithm)
            obs, reward, terminated, truncated, info = eval_env.step(action)
            done = terminated or truncated
            total_reward += reward

        rows.append({
            "algorithm": algorithm,
            "episode": episode + 1,
            "reward": total_reward,
            "steps": int(info.get("steps", 0)),
            "final_distance": float(info.get("dist_to_goal", np.nan)),
            "collision": bool(info.get("collision", False)),
            "goal_reached": bool(info.get("goal_reached", False)),
        })

    eval_env.close()
    return rows

def rows_to_markdown(rows):
    header = "| Algorithm | Episode | Reward | Steps | Final distance | Collision | Goal reached |"
    separator = "|---|---:|---:|---:|---:|:---:|:---:|"
    body = []
    for row in rows:
        body.append(
            f"| {row['algorithm']} | {row['episode']} | {row['reward']:.2f} | {row['steps']} | "
            f"{row['final_distance']:.2f} | {row['collision']} | {row['goal_reached']} |"
        )
    return "\n".join([header, separator, *body])

def success_summary(rows):
    lines = []
    for algorithm in sorted({row["algorithm"] for row in rows}):
        algo_rows = [row for row in rows if row["algorithm"] == algorithm]
        success_rate = np.mean([row["goal_reached"] for row in algo_rows])
        mean_reward = np.mean([row["reward"] for row in algo_rows])
        mean_steps = np.mean([row["steps"] for row in algo_rows])
        lines.append(f"- {algorithm}: success rate {success_rate:.2f}, mean reward {mean_reward:.2f}, mean steps {mean_steps:.1f}")
    return "\n".join(lines)

## Run Comparison

In [5]:
ddpg_rows = run_recorded_benchmark(
    agent        = ddpg_agent,
    algorithm    = "DDPG",
    env_kwargs   = ENV_KWARGS,
    video_folder = COMPARISON_VIDEO_DIR,
    name_prefix  = DDPG_NAME_PREFIX,
    n_episodes   = N_EPISODES,
)

sac_rows = run_recorded_benchmark(
    agent        = sac_agent,
    algorithm    = "SAC",
    env_kwargs   = ENV_KWARGS,
    video_folder = COMPARISON_VIDEO_DIR,
    name_prefix  = SAC_NAME_PREFIX,
    n_episodes   = N_EPISODES,
)

comparison_rows = ddpg_rows + sac_rows
display(Markdown(rows_to_markdown(comparison_rows)))
display(Markdown(success_summary(comparison_rows)))

c:\Users\39324\anaconda3\envs\naml_libraries\lib\site-packages\gymnasium\wrappers\rendering.py:293: UserWarning: WARN: Overwriting existing videos at c:\Users\39324\Desktop\NAML_RL_gym\Continuous_Diff_Drive\videos\comparison folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


| Algorithm | Episode | Reward | Steps | Final distance | Collision | Goal reached |
|---|---:|---:|---:|---:|:---:|:---:|
| DDPG | 1 | 190.29 | 1000 | 7.82 | False | False |
| DDPG | 2 | 190.29 | 1000 | 7.82 | False | False |
| DDPG | 3 | 190.29 | 1000 | 7.82 | False | False |
| SAC | 1 | 591.16 | 127 | 0.50 | False | True |
| SAC | 2 | 591.16 | 127 | 0.50 | False | True |
| SAC | 3 | 591.16 | 127 | 0.50 | False | True |

- DDPG: success rate 0.00, mean reward 190.29, mean steps 1000.0
- SAC: success rate 1.00, mean reward 591.16, mean steps 127.0

## Comparison Videos

In [6]:
for prefix in (DDPG_NAME_PREFIX, SAC_NAME_PREFIX):
    print(prefix)
    video_paths = sorted(COMPARISON_VIDEO_DIR.glob(f"{prefix}*.mp4"))
    if not video_paths:
        print(f"No videos found for {prefix}")
    for video_path in video_paths:
        print(video_path.name)
        display(Video(filename=str(video_path), embed=True))

compare_ddpg_diff_drive_eval_fixed_obstacles_greedy
compare_ddpg_diff_drive_eval_fixed_obstacles_greedy-episode-0.mp4


compare_ddpg_diff_drive_eval_fixed_obstacles_greedy-episode-1.mp4


compare_ddpg_diff_drive_eval_fixed_obstacles_greedy-episode-2.mp4


compare_sac_diff_drive_eval_fixed_obstacles_greedy
compare_sac_diff_drive_eval_fixed_obstacles_greedy-episode-0.mp4


compare_sac_diff_drive_eval_fixed_obstacles_greedy-episode-1.mp4


compare_sac_diff_drive_eval_fixed_obstacles_greedy-episode-2.mp4
